In [5]:
import os
import numpy as np
import pandas as pd


IN_DIR = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features"
OUT_DIR = os.path.join(IN_DIR, "rank_diff_selected_models_top15")
os.makedirs(OUT_DIR, exist_ok=True)

TOP_N = 15

traits = [
    "Yield_per_plant",
    "1000-grain_weight",
    "Grain_length_to_width_ratio",
    "Heading_date",
    "Plant_height",
    "Grain_per_panicle",
    "Seed_set_rate",
    "Tiller_number"
]

In [6]:


summary_records = []

for trait in traits:
    in_file = os.path.join(IN_DIR, f"{trait}_ALL_features_rank_matrix.tsv")

    if not os.path.exists(in_file):
        print(f"Skip {trait}: file not found -> {in_file}")
        continue

    df = pd.read_table(in_file)

    meta_cols = ["model_name", "year", "location"]
    feature_cols = [c for c in df.columns if c not in meta_cols]

    fitall_df = df[df["model_name"] == "CB_FitAll"]
    dynamic_df = df[df["model_name"].str.startswith("CB_Dynamic", na=False)]

    if fitall_df.empty or dynamic_df.empty:
        print(f"Skip {trait}: missing CB_FitAll or CB_Dynamic rows")
        continue

    fitall_rank_all = fitall_df.iloc[0][feature_cols].astype(float)

    # 所有特征仍然按照 CB_FitAll 的 rank 从小到大排序，用于最终输出
    ordered_features_all = fitall_rank_all.sort_values().index.tolist()
    fitall_rank_all = fitall_rank_all[ordered_features_all]

    # 只用 CB_FitAll 排名前 TOP_N 的特征计算 rank 差异
    top_features = ordered_features_all[:TOP_N]
    fitall_rank_top = fitall_rank_all[top_features]

    diff_records = []

    for idx, row in dynamic_df.iterrows():
        dynamic_rank_top = row[top_features].astype(float)

        mean_abs_diff_top15 = np.nanmean(
            np.abs(dynamic_rank_top - fitall_rank_top)
        )

        diff_records.append({
            "idx": idx,
            "model_name": row["model_name"],
            "year": row["year"],
            "location": row["location"],
            "mean_abs_rank_diff_top15": mean_abs_diff_top15
        })

    diff_df = pd.DataFrame(diff_records)

    min_row = diff_df.loc[diff_df["mean_abs_rank_diff_top15"].idxmin()]
    max_row = diff_df.loc[diff_df["mean_abs_rank_diff_top15"].idxmax()]

    # 输出仍然保留所有特征的 rank
    min_model_rank_all = df.loc[min_row["idx"], ordered_features_all].astype(float)
    max_model_rank_all = df.loc[max_row["idx"], ordered_features_all].astype(float)

    out_df = pd.DataFrame({
        "feature": ordered_features_all,
        "CB_FitAll_rank": fitall_rank_all.values,
        f"{min_row['model_name']}_rank": min_model_rank_all.values,
        f"{max_row['model_name']}_rank": max_model_rank_all.values
    })

    out_file = os.path.join(
        OUT_DIR,
        f"{trait}_FitAll_MinDiff_MaxDiff_top15_rank_all_features.tsv"
    )
    out_df.to_csv(out_file, sep="\t", index=False)

    summary_records.append({
        "Trait": trait,
        "diff_used_top_n_features": TOP_N,
        "min_diff_model": min_row["model_name"],
        "min_diff_year": min_row["year"],
        "min_diff_location": min_row["location"],
        "min_mean_abs_rank_diff_top15": min_row["mean_abs_rank_diff_top15"],
        "max_diff_model": max_row["model_name"],
        "max_diff_year": max_row["year"],
        "max_diff_location": max_row["location"],
        "max_mean_abs_rank_diff_top15": max_row["mean_abs_rank_diff_top15"],
        "output_file": out_file
    })

    print(f"Saved: {out_file}")


summary_df = pd.DataFrame(summary_records)

summary_file = os.path.join(
    OUT_DIR,
    "ALL_traits_min_max_rank_diff_top15_summary.tsv"
)
summary_df.to_csv(summary_file, sep="\t", index=False)

print("\nAll done.")
print(f"Saved summary: {summary_file}")

Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/rank_diff_selected_models_top15/Yield_per_plant_FitAll_MinDiff_MaxDiff_top15_rank_all_features.tsv
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/rank_diff_selected_models_top15/1000-grain_weight_FitAll_MinDiff_MaxDiff_top15_rank_all_features.tsv
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/rank_diff_selected_models_top15/Grain_length_to_width_ratio_FitAll_MinDiff_MaxDiff_top15_rank_all_features.tsv
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/rank_diff_selected_models_top15/Heading_date_FitAll_MinDiff_MaxDiff_top15_rank_all_features.tsv
Saved: /data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_rank_all_features/rank_diff_selected_models_top15/Plant_height_FitAll_MinDiff_MaxDiff_top15_rank_all_features.tsv
Saved: /data2/zhoujb/project/cro